[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/255ribeiro/cadquery-simpleViewer/blob/main/notebooks/export_exploration_build123d.ipynb)

# Caderno de exploração (build123d, modo álgebra): exportação (`export`, `export_ifc`, `ifc_config`)

Baseado no caderno [`interactive_exploration_build123d.ipynb`](interactive_exploration_build123d.ipynb), mas focado especificamente nos recursos de exportação de `show()`/`@interactive()`: o dropdown+botão **Export** (formatos STEP e IFC Proxy), os dicionários de configuração `export`/`export_ifc` (nome do arquivo, unidade) e `ifc_config` (versão do schema IFC), além de chamadas diretas a `export_step()`/`export_ifc_proxy()` para conferir o resultado sem precisar clicar no botão. Não faz parte da suíte automatizada de testes — execute as células interativamente.

## Configuração no Google Colab

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "cadquery-simpleviewer[build123d,interactive,ifc] @ "
         "git+https://github.com/255ribeiro/cadquery-simpleViewer.git@main"],
        check=True,
    )
    # build123d traz um ipython mais novo do que o bootstrap do kernel do
    # Colab tolera. Devolve a versão compatível para o disco — NÃO reinicie
    # o runtime, o kernel atual já está com o ipython funcional carregado.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "ipython==7.34.0", "--no-deps"],
        check=True,
    )
else:
    print("Não é o Colab, pulando a instalação de pacotes.")

In [ ]:
import os
import tempfile

from build123d import Box, Location

from cadquery_simpleviewer import show, interactive
from cadquery_simpleviewer.exporter import export_step, resolve_export_config
from cadquery_simpleviewer.ifc_exporter import (
    export_ifc_proxy, resolve_ifc_export_config, resolve_ifc_config,
)

try:
    import ifcopenshell
    IFC_DISPONIVEL = True
except ImportError:
    IFC_DISPONIVEL = False
    print("ifcopenshell não instalado — instale o extra [ifc] para testar a exportação IFC "
          "(a opção 'IFC Proxy' simplesmente não aparecerá no dropdown Export).")

## 1. `show()` — exportação padrão (dropdown STEP + IFC Proxy)

Por padrão `export=None` e `export_ifc=None` deixam as duas opções habilitadas — o dropdown **Export**, alinhado à direita acima do gráfico, mostra "STEP" (selecionado por padrão) e "IFC Proxy" (se `ifcopenshell` estiver instalado). Escolha um formato e clique em **Export** para escrever `model.step`/`model.ifc` no diretório de trabalho atual.

In [ ]:
caixa = Box(5, 3, 2)
show(caixa)

## 1.1 Eixos locais e eixos do mundo (`objects` com `Location`/`Plane`/`Axis`, `world_axes=`)

Passar um `Location`, `Plane` ou `Axis` do build123d na lista de `objects` desenha um tripé RGB (X vermelho, Y verde, Z azul) na origem daquele objeto — desenhado como cilindros sólidos (não linhas finas), para renderizar de forma confiável em qualquer navegador, incluindo o Colab. Os botões **Local Axes** e **Origin** (liga/desliga o tripé fixo em `(0, 0, 0)`, controlado por `world_axes=`) aparecem sempre no cabeçalho, mesmo sem nenhum `Location` passado — ambos começam **desligados** por padrão (`local_axes_visible=False`, `world_axes=False`): ligar funciona de forma confiável, mas desligar um tripé já exibido pode não atualizar a view em todo navegador (limitação conhecida do Plotly ao ocultar `Mesh3d` dinamicamente), então o padrão evita depender dessa direção.

`axes_scale=` controla o comprimento do braço do tripé (padrão `1`) — se ele for menor que a metade das dimensões do sólido, o tripé fica inteiramente dentro da geometria opaca e some visualmente. Escolha um valor maior que a metade da maior dimensão do objeto.

Movemos a caixa para longe da origem do mundo com `.moved(Location(...))` de propósito: se o objeto ficar centrado em `(0, 0, 0)`, o tripé local e o tripé do mundo coincidem exatamente, e o tripé do mundo (opaco) vence o "z-fighting" contra o tripé local (mais transparente) — parecendo que o local nunca apareceu. Com a caixa deslocada, os dois tripés ficam visualmente distintos: um preso à caixa, outro fixo na origem.

In [ ]:
caixa_deslocada = caixa.moved(Location((6, 4, 0)))
show([caixa_deslocada, caixa_deslocada.location], world_axes=True, local_axes_visible=True, axes_scale=4)

## 2. Personalizando o STEP (`export=`)

`export` aceita um dict com `filename` e `unit` (`"M"` ou `"MM"`), ou `False` para remover a opção "STEP" do dropdown.

In [ ]:
show(caixa, export=dict(filename="saida/caixa.step", unit="MM"))

## 3. Personalizando o IFC (`export_ifc=`)

Mesmo contrato de `export`, mas para a opção "IFC Proxy": `filename` e `unit`. Cada sólido é exportado como seu próprio `IfcBuildingElementProxy` (não um compound único como no STEP).

In [ ]:
show(caixa, export_ifc=dict(filename="saida/caixa.ifc", unit="MM"))

## 4. Escolhendo a versão do schema IFC (`ifc_config=`)

`ifc_config=dict(schema=...)` controla qual schema IFC é escrito — o padrão é `"IFC4"`. Ferramentas mais antigas podem exigir `"IFC2X3"`.

In [ ]:
show(caixa, ifc_config=dict(schema="IFC2X3"))

## 5. Desabilitando formatos (`export=False` / `export_ifc=False`)

Desabilitar um formato apenas remove sua opção do dropdown — desabilitar os dois remove a linha de exportação inteira.

In [ ]:
print("Só 'IFC Proxy' no dropdown (STEP desabilitado):")
show(caixa, export=False)

In [ ]:
print("Só 'STEP' no dropdown (IFC desabilitado):")
show(caixa, export_ifc=False)

In [ ]:
print("Nenhuma linha de exportação (os dois desabilitados):")
show(caixa, export=False, export_ifc=False)

## 6. `@interactive()` com exportação configurada via `show_kwargs`

`export`, `export_ifc` e `ifc_config` são passados dentro do dicionário `show_kwargs` do decorador — nunca como argumentos soltos, para não colidirem com o nome de um parâmetro do modelo. Clicar em Export sempre exporta o objeto construído com os valores **atuais** dos sliders.

In [ ]:
@interactive(
    width=(1, 10, 0.5, 5),
    height=(1, 8, 0.5, 3),
    show_kwargs=dict(
        export=dict(filename="saida/interativo.step"),
        export_ifc=dict(filename="saida/interativo.ifc"),
        ifc_config=dict(schema="IFC2X3"),
    ),
)
def modelo_exportavel(width, height):
    return Box(width, height, 2)

## 7. Testando programaticamente (sem clicar no botão)

`export_step()`/`export_ifc_proxy()` podem ser chamadas diretamente — é como o botão Export funciona por baixo dos panos. Útil para conferir o resultado sem interação manual: exporta para um diretório temporário, reabre o IFC com `ifcopenshell` e imprime o que foi escrito.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    step_path = export_step(caixa, os.path.join(tmp, "caixa.step"), unit="M")
    print("STEP exportado:", step_path, "-", os.path.getsize(step_path), "bytes")

    if IFC_DISPONIVEL:
        ifc_path = export_ifc_proxy(caixa, os.path.join(tmp, "caixa.ifc"), unit="M", schema="IFC4")
        print("IFC exportado:", ifc_path, "-", os.path.getsize(ifc_path), "bytes")

        model = ifcopenshell.open(ifc_path)
        print("schema:", model.schema)
        print("proxies:", [p.Name for p in model.by_type("IfcBuildingElementProxy")])
        print("propsets (deve ser vazio — só geometria e nome):",
              model.by_type("IfcPropertySet"))
    else:
        print("ifcopenshell não instalado — pulando a exportação IFC.")

## 8. Múltiplos objetos — um proxy IFC por sólido, um compound único no STEP

Diferente do STEP (que combina vários sólidos em um único compound), o IFC gera um `IfcBuildingElementProxy` independente por objeto, mantendo cada um selecionável separadamente no programa BIM — e usa `names` para nomear cada proxy.

In [ ]:
caixa_a = Box(5, 3, 2)
caixa_b = Box(2, 2, 2)

if IFC_DISPONIVEL:
    with tempfile.TemporaryDirectory() as tmp:
        ifc_path = export_ifc_proxy(
            [caixa_a, caixa_b], os.path.join(tmp, "duas_caixas.ifc"),
            names=["Caixa A", "Caixa B"],
        )
        model = ifcopenshell.open(ifc_path)
        print([p.Name for p in model.by_type("IfcBuildingElementProxy")])
else:
    print("ifcopenshell não instalado — pulando a exportação IFC.")

In [ ]:
show([caixa_a, caixa_b], names=["Caixa A", "Caixa B"])

## 9. Schema IFC inválido (erro esperado)

Um identificador de schema que o `ifcopenshell` instalado não reconhece levanta um erro (o tipo exato depende da versão do `ifcopenshell` — normalmente `RuntimeError`).

In [ ]:
if IFC_DISPONIVEL:
    try:
        export_ifc_proxy(caixa, "invalido.ifc", schema="NAO_EXISTE")
    except Exception as e:
        print(f"{type(e).__name__}: {e}")
else:
    print("ifcopenshell não instalado — pulando este teste.")

## 10. Conferindo os resolvers de configuração

`resolve_export_config()`, `resolve_ifc_export_config()` e `resolve_ifc_config()` normalizam `export`/`export_ifc`/`ifc_config` (o `None`/`False`/`dict` que `show()` recebe) para os dicts efetivamente usados na exportação — útil para conferir rapidamente qual configuração está sendo aplicada.

In [ ]:
print("export=None       ->", resolve_export_config(None))
print("export=False      ->", resolve_export_config(False))
print("export_ifc=None   ->", resolve_ifc_export_config(None))
print("export_ifc=dict   ->", resolve_ifc_export_config(dict(unit="MM")))
print("ifc_config=None   ->", resolve_ifc_config(None))
print("ifc_config=dict   ->", resolve_ifc_config(dict(schema="IFC2X3")))